**© Copyright AIDENTIFY. All rights reserved.**

# Part 3 | Session 22: 양자화 (Quantization) 개념

## 🎯 양자화(Quantization) 개요

양자화는 모델의 가중치(weight)를 더 낮은 정밀도(precision)로 변환하여 **메모리 사용량을 줄이고 추론 속도를 높이는** 핵심 기술입니다.

### 이번 세션에서 배울 내용

- 1️⃣ 양자화의 필요성 (메모리, 속도, 비용)
- 2️⃣ 정밀도별 비교 (FP32 → FP16 → INT8 → INT4)
- 3️⃣ BitsAndBytes 양자화 실습 (4bit, 8bit)
- 4️⃣ GPTQ 양자화 개요 및 실습
- 5️⃣ AWQ 양자화 개요
- 6️⃣ GGUF 포맷과 llama.cpp
- 7️⃣ 양자화 방법별 성능 비교 (메모리, 속도, 품질)
- 8️⃣ RTX 4060에서의 최적 양자화 전략

### 실습 환경
- **GPU**: 필수 (RTX 4060 8GB 또는 TITAN RTX 24GB)
- **모델**: Qwen2.5-1.5B-Instruct, Qwen2.5-3B-Instruct
- **핵심 패키지**: transformers, bitsandbytes, auto-gptq, autoawq

In [ ]:
# 🔧 필수 패키지 설치 (필요 시 주석 해제)
# !pip install transformers accelerate bitsandbytes
# !pip install auto-gptq  # GPTQ 양자화용
# !pip install autoawq    # AWQ 양자화용

print("✅ 패키지 설치 완료!")

In [ ]:
# 📦 기본 라이브러리 임포트
import torch
import gc
import time
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"총 VRAM: {total_vram:.1f}GB")

In [ ]:
# 🔍 GPU 메모리 모니터링 유틸리티
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

def cleanup():
    """GPU 메모리 정리"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 메모리 정리 완료")
    print_gpu_memory("정리 후")

print_gpu_memory("초기 상태")
print("✅ GPU 모니터링 함수 정의 완료!")

In [ ]:
# 📌 실습에 사용할 모델 설정
MODEL_NAME_1_5B = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_NAME_3B = "Qwen/Qwen2.5-3B-Instruct"

# 테스트용 프롬프트
TEST_PROMPT = "인공지능의 미래에 대해 3문장으로 설명해주세요."

print(f"📌 1.5B 모델: {MODEL_NAME_1_5B}")
print(f"📌 3B 모델: {MODEL_NAME_3B}")
print(f"📌 테스트 프롬프트: {TEST_PROMPT}")

---

## 1️⃣ 양자화의 필요성

### 왜 양자화가 필요한가?

LLM은 수십억 개의 파라미터를 가지고 있어 **메모리**와 **연산 비용**이 매우 큽니다.

| 문제 | 설명 |
|------|------|
| 💾 **메모리** | 7B 모델 FP32 = 약 28GB VRAM 필요 |
| ⚡ **속도** | 높은 정밀도 = 느린 연산 |
| 💰 **비용** | GPU 비용이 정밀도에 비례 |
| 🖥️ **접근성** | 소비자용 GPU에서 실행 불가능 |

### 양자화로 해결!

- 🎯 **메모리 절감**: FP32 → INT4로 **약 8배** 메모리 절약
- 🎯 **속도 향상**: 더 작은 데이터 타입 = 더 빠른 연산
- 🎯 **접근성 향상**: RTX 4060(8GB)에서도 7B 모델 실행 가능
- 🎯 **품질 유지**: 적절한 양자화 방법 사용 시 성능 저하 미미

In [ ]:
# 📊 모델 크기별 메모리 요구량 계산

def calculate_model_memory(params_billion, precision_bits):
    """모델의 메모리 사용량을 계산합니다.
    
    Args:
        params_billion: 파라미터 수 (10억 단위)
        precision_bits: 정밀도 비트 수
    
    Returns:
        메모리 사용량 (GB)
    """
    bytes_per_param = precision_bits / 8
    memory_gb = params_billion * 1e9 * bytes_per_param / 1024**3
    return memory_gb

# 다양한 모델 크기와 정밀도에 대한 메모리 계산
model_sizes = [1.5, 3, 7, 13, 70]
precisions = {
    'FP32 (32bit)': 32,
    'FP16 (16bit)': 16,
    'INT8 (8bit)': 8,
    'INT4 (4bit)': 4
}

print("📊 모델 크기별 / 정밀도별 메모리 요구량 (GB)")
print("=" * 70)
header = f"{'모델':>8s} | " + " | ".join(f"{p:>14s}" for p in precisions.keys())
print(header)
print("-" * 70)

for size in model_sizes:
    row = f"{size:>6.1f}B | "
    values = []
    for name, bits in precisions.items():
        mem = calculate_model_memory(size, bits)
        values.append(f"{mem:>12.1f}GB")
    row += " | ".join(values)
    print(row)

print("=" * 70)
print("\n💡 참고: 실제 추론 시에는 KV Cache 등 추가 메모리가 필요합니다.")
print("💡 RTX 4060 (8GB): INT4 양자화 시 7B 모델까지 가능")
print("💡 TITAN RTX (24GB): FP16으로 7B, INT4로 13B 모델 가능")

---

## 2️⃣ 정밀도별 비교 (FP32 → FP16 → INT8 → INT4)

### 숫자 표현 방식

| 정밀도 | 비트 수 | 범위 | 용도 |
|--------|---------|------|------|
| **FP32** | 32비트 | ±3.4×10³⁸ | 학습 기본 정밀도 |
| **FP16** | 16비트 | ±65,504 | Mixed Precision 학습 |
| **BF16** | 16비트 | ±3.4×10³⁸ | 넓은 범위, 낮은 정밀도 |
| **INT8** | 8비트 | -128 ~ 127 | 추론 최적화 |
| **INT4** | 4비트 | -8 ~ 7 | 극한 압축 |

### 양자화 방식

- 🔹 **Post-Training Quantization (PTQ)**: 학습 후 양자화 (BitsAndBytes, GPTQ, AWQ)
- 🔹 **Quantization-Aware Training (QAT)**: 학습 중 양자화 고려
- 🔹 **Dynamic Quantization**: 추론 시 동적으로 양자화

In [ ]:
# 🔢 정밀도별 숫자 표현 차이 시뮬레이션
import numpy as np

print("🔢 정밀도별 숫자 표현 비교")
print("=" * 60)

# FP32 원본 값
original_value = 3.141592653589793
print(f"\n원본 값 (FP64): {original_value}")

# FP32
fp32_val = np.float32(original_value)
print(f"FP32:           {fp32_val} (오차: {abs(original_value - fp32_val):.2e})")

# FP16
fp16_val = np.float16(original_value)
print(f"FP16:           {fp16_val} (오차: {abs(original_value - float(fp16_val)):.2e})")

# INT8 시뮬레이션 (스케일링)
# 범위를 -1 ~ 1로 가정하고 INT8로 변환
scale = 127.0  # INT8 범위
int8_val = round(original_value / 4 * scale) / scale * 4  # 0~4 범위 스케일링
print(f"INT8 (근사):    {int8_val:.6f} (오차: {abs(original_value - int8_val):.2e})")

# INT4 시뮬레이션
scale_4bit = 7.0  # INT4 범위
int4_val = round(original_value / 4 * scale_4bit) / scale_4bit * 4
print(f"INT4 (근사):    {int4_val:.6f} (오차: {abs(original_value - int4_val):.2e})")

print("\n💡 낮은 정밀도일수록 오차가 증가하지만, LLM에서는 대부분 허용 가능한 수준입니다.")

In [ ]:
# 📊 양자화에 따른 가중치 분포 변화 시각화
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# 가상의 모델 가중치 생성 (정규분포)
np.random.seed(42)
weights = np.random.normal(0, 0.5, 10000).astype(np.float32)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# FP32
axes[0].hist(weights, bins=100, color='blue', alpha=0.7)
axes[0].set_title('FP32 (Original)', fontsize=12)
axes[0].set_xlabel('Weight Value')
axes[0].set_ylabel('Count')

# FP16
fp16_weights = weights.astype(np.float16).astype(np.float32)
axes[1].hist(fp16_weights, bins=100, color='green', alpha=0.7)
axes[1].set_title('FP16', fontsize=12)
axes[1].set_xlabel('Weight Value')

# INT8 시뮬레이션
w_min, w_max = weights.min(), weights.max()
scale_8 = (w_max - w_min) / 255
int8_weights = np.round((weights - w_min) / scale_8) * scale_8 + w_min
axes[2].hist(int8_weights, bins=100, color='orange', alpha=0.7)
axes[2].set_title('INT8 (Simulated)', fontsize=12)
axes[2].set_xlabel('Weight Value')

# INT4 시뮬레이션
scale_4 = (w_max - w_min) / 15
int4_weights = np.round((weights - w_min) / scale_4) * scale_4 + w_min
axes[3].hist(int4_weights, bins=16, color='red', alpha=0.7)
axes[3].set_title('INT4 (Simulated)', fontsize=12)
axes[3].set_xlabel('Weight Value')

plt.suptitle('Quantization: Weight Distribution Changes', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('quantization_distributions.png', bbox_inches='tight', dpi=100)
plt.show()

print("✅ 양자화 단계별 가중치 분포 시각화 완료!")
print("💡 INT4에서는 가중치가 16개 값으로만 표현됩니다 → 이산화(discretization)")

---

## 3️⃣ BitsAndBytes 양자화 실습 (4bit, 8bit)

**BitsAndBytes**는 가장 간편하게 사용할 수 있는 양자화 라이브러리입니다.

### 특징
- 🔹 **HuggingFace Transformers와 완벽 통합**
- 🔹 `BitsAndBytesConfig`로 간편 설정
- 🔹 8bit (LLM.int8()) 및 4bit (QLoRA용 NF4) 지원
- 🔹 **동적 양자화**: 모델 로딩 시 자동 변환
- 🔹 QLoRA 학습 시 필수 라이브러리

### 주요 옵션
- `load_in_8bit`: INT8 양자화
- `load_in_4bit`: INT4 양자화
- `bnb_4bit_quant_type`: 양자화 타입 (nf4 / fp4)
- `bnb_4bit_compute_dtype`: 연산 시 사용할 dtype
- `bnb_4bit_use_double_quant`: 이중 양자화 (메모리 추가 절약)

In [ ]:
# 📏 Step 1: FP16 모델 로딩 (기준선)
print("📏 FP16 모델 로딩 중... (기준선 측정)")
print_gpu_memory("로딩 전")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_1_5B)

start_time = time.time()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_1_5B,
    torch_dtype=torch.float16,
    device_map="auto"
)
load_time_fp16 = time.time() - start_time

print_gpu_memory("FP16 로딩 후")
print(f"⏱️ FP16 로딩 시간: {load_time_fp16:.2f}초")
print(f"📊 모델 파라미터 수: {sum(p.numel() for p in model_fp16.parameters()) / 1e9:.2f}B")

In [ ]:
# ⚡ FP16 모델 추론 속도 측정
print("⚡ FP16 모델 추론 속도 측정 중...")

messages = [{"role": "user", "content": TEST_PROMPT}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model_fp16.device)

# 워밍업
with torch.no_grad():
    _ = model_fp16.generate(**inputs, max_new_tokens=10)

# 실제 측정
start_time = time.time()
with torch.no_grad():
    outputs_fp16 = model_fp16.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )
inference_time_fp16 = time.time() - start_time

generated_text_fp16 = tokenizer.decode(outputs_fp16[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
num_tokens_fp16 = outputs_fp16.shape[1] - inputs['input_ids'].shape[1]

print(f"\n📝 FP16 생성 결과:")
print(f"  {generated_text_fp16}")
print(f"\n⏱️ 생성 시간: {inference_time_fp16:.2f}초")
print(f"📊 생성 토큰 수: {num_tokens_fp16}")
print(f"⚡ 토큰/초: {num_tokens_fp16/inference_time_fp16:.1f} tokens/sec")

# FP16 결과 저장
fp16_results = {
    'memory': torch.cuda.memory_allocated() / 1024**3,
    'load_time': load_time_fp16,
    'inference_time': inference_time_fp16,
    'tokens_per_sec': num_tokens_fp16 / inference_time_fp16,
    'text': generated_text_fp16
}

In [ ]:
# 🧹 FP16 모델 메모리 해제
del model_fp16
del outputs_fp16
cleanup()

In [ ]:
# 🔧 Step 2: 8bit 양자화 모델 로딩
print("🔧 8bit 양자화 모델 로딩 중...")
print_gpu_memory("로딩 전")

quantization_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True  # INT8 양자화
)

start_time = time.time()
model_8bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_1_5B,
    quantization_config=quantization_config_8bit,
    device_map="auto"
)
load_time_8bit = time.time() - start_time

print_gpu_memory("8bit 로딩 후")
print(f"⏱️ 8bit 로딩 시간: {load_time_8bit:.2f}초")

In [ ]:
# ⚡ 8bit 모델 추론 속도 측정
print("⚡ 8bit 모델 추론 속도 측정 중...")

inputs = tokenizer(text, return_tensors="pt").to(model_8bit.device)

# 워밍업
with torch.no_grad():
    _ = model_8bit.generate(**inputs, max_new_tokens=10)

# 실제 측정
start_time = time.time()
with torch.no_grad():
    outputs_8bit = model_8bit.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )
inference_time_8bit = time.time() - start_time

generated_text_8bit = tokenizer.decode(outputs_8bit[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
num_tokens_8bit = outputs_8bit.shape[1] - inputs['input_ids'].shape[1]

print(f"\n📝 8bit 생성 결과:")
print(f"  {generated_text_8bit}")
print(f"\n⏱️ 생성 시간: {inference_time_8bit:.2f}초")
print(f"📊 생성 토큰 수: {num_tokens_8bit}")
print(f"⚡ 토큰/초: {num_tokens_8bit/inference_time_8bit:.1f} tokens/sec")

# 8bit 결과 저장
int8_results = {
    'memory': torch.cuda.memory_allocated() / 1024**3,
    'load_time': load_time_8bit,
    'inference_time': inference_time_8bit,
    'tokens_per_sec': num_tokens_8bit / inference_time_8bit,
    'text': generated_text_8bit
}

# 메모리 정리
del model_8bit, outputs_8bit
cleanup()

In [ ]:
# 🔧 Step 3: 4bit 양자화 모델 로딩 (NF4 - QLoRA 방식)
print("🔧 4bit NF4 양자화 모델 로딩 중...")
print_gpu_memory("로딩 전")

quantization_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,                          # 4bit 양자화
    bnb_4bit_quant_type="nf4",                  # NormalFloat4 (QLoRA 논문)
    bnb_4bit_compute_dtype=torch.bfloat16,        # 연산 시 FP16 사용
    bnb_4bit_use_double_quant=True               # 이중 양자화 (추가 메모리 절약)
)

print("\n📋 4bit 양자화 설정:")
print(f"  - 양자화 타입: NF4 (NormalFloat4)")
print(f"  - 연산 dtype: float16")
print(f"  - 이중 양자화: True")

start_time = time.time()
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_1_5B,
    quantization_config=quantization_config_4bit,
    device_map="auto"
)
load_time_4bit = time.time() - start_time

print_gpu_memory("4bit 로딩 후")
print(f"⏱️ 4bit 로딩 시간: {load_time_4bit:.2f}초")

In [ ]:
# ⚡ 4bit 모델 추론 속도 측정
print("⚡ 4bit 모델 추론 속도 측정 중...")

inputs = tokenizer(text, return_tensors="pt").to(model_4bit.device)

# 워밍업
with torch.no_grad():
    _ = model_4bit.generate(**inputs, max_new_tokens=10)

# 실제 측정
start_time = time.time()
with torch.no_grad():
    outputs_4bit = model_4bit.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )
inference_time_4bit = time.time() - start_time

generated_text_4bit = tokenizer.decode(outputs_4bit[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
num_tokens_4bit = outputs_4bit.shape[1] - inputs['input_ids'].shape[1]

print(f"\n📝 4bit 생성 결과:")
print(f"  {generated_text_4bit}")
print(f"\n⏱️ 생성 시간: {inference_time_4bit:.2f}초")
print(f"📊 생성 토큰 수: {num_tokens_4bit}")
print(f"⚡ 토큰/초: {num_tokens_4bit/inference_time_4bit:.1f} tokens/sec")

# 4bit 결과 저장
int4_results = {
    'memory': torch.cuda.memory_allocated() / 1024**3,
    'load_time': load_time_4bit,
    'inference_time': inference_time_4bit,
    'tokens_per_sec': num_tokens_4bit / inference_time_4bit,
    'text': generated_text_4bit
}

# 메모리 정리
del model_4bit, outputs_4bit
cleanup()

In [ ]:
# 📊 BitsAndBytes 양자화 결과 비교
print("📊 BitsAndBytes 양자화 결과 비교 (Qwen2.5-1.5B-Instruct)")
print("=" * 70)
print(f"{'항목':>15s} | {'FP16':>12s} | {'INT8':>12s} | {'INT4 (NF4)':>12s}")
print("-" * 70)
print(f"{'GPU 메모리':>15s} | {fp16_results['memory']:>10.2f}GB | {int8_results['memory']:>10.2f}GB | {int4_results['memory']:>10.2f}GB")
print(f"{'로딩 시간':>15s} | {fp16_results['load_time']:>10.2f}초 | {int8_results['load_time']:>10.2f}초 | {int4_results['load_time']:>10.2f}초")
print(f"{'추론 시간':>15s} | {fp16_results['inference_time']:>10.2f}초 | {int8_results['inference_time']:>10.2f}초 | {int4_results['inference_time']:>10.2f}초")
print(f"{'토큰/초':>15s} | {fp16_results['tokens_per_sec']:>10.1f}  | {int8_results['tokens_per_sec']:>10.1f}  | {int4_results['tokens_per_sec']:>10.1f} ")
print("=" * 70)

# 메모리 절감율 계산
mem_reduction_8bit = (1 - int8_results['memory'] / fp16_results['memory']) * 100
mem_reduction_4bit = (1 - int4_results['memory'] / fp16_results['memory']) * 100
print(f"\n💾 메모리 절감율:")
print(f"  - INT8: FP16 대비 {mem_reduction_8bit:.1f}% 절감")
print(f"  - INT4: FP16 대비 {mem_reduction_4bit:.1f}% 절감")

print("\n📝 생성 결과 비교:")
print(f"  FP16: {fp16_results['text'][:80]}...")
print(f"  INT8: {int8_results['text'][:80]}...")
print(f"  INT4: {int4_results['text'][:80]}...")

---

## 4️⃣ GPTQ 양자화 개요 및 실습

**GPTQ (GPT Quantization)**는 사후 훈련 양자화 기법 중 가장 널리 사용되는 방법입니다.

### GPTQ 특징
- 🔹 **Calibration 데이터 기반**: 소량의 데이터로 양자화 오차 최소화
- 🔹 **레이어 단위 양자화**: 각 레이어의 가중치를 순차적으로 양자화
- 🔹 **Hessian 기반 최적화**: 중요한 가중치를 더 정밀하게 보존
- 🔹 **오프라인 양자화**: 한 번 양자화하면 바로 사용 가능
- 🔹 **GPU 커널 최적화**: ExLlama, Marlin 커널로 빠른 추론

### BitsAndBytes vs GPTQ

| 항목 | BitsAndBytes | GPTQ |
|------|-------------|------|
| 양자화 시점 | 로딩 시 동적 | 사전 양자화 (오프라인) |
| 속도 | 보통 | 빠름 (최적화 커널) |
| 품질 | 좋음 | 약간 더 좋음 (캘리브레이션) |
| 사용 편의성 | 매우 쉬움 | 양자화 과정 필요 |
| QLoRA 학습 | 지원 | 지원 (제한적) |

In [ ]:
# 🔧 GPTQ 양자화 실습
# 참고: GPTQ 양자화는 시간이 걸리므로, 이미 양자화된 모델을 로딩하는 방법도 함께 설명합니다.

print("🔧 GPTQ 양자화 실습")
print("=" * 60)

# 방법 1: 직접 양자화 (시간이 걸림)
print("\n📋 방법 1: 직접 GPTQ 양자화 코드 (참고용)")
print("-" * 60)

gptq_code = '''
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig

# 1. 토크나이저 로딩
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

# 2. GPTQ 양자화 설정
gptq_config = GPTQConfig(
    bits=4,                      # 4bit 양자화
    dataset="c4",                # 캘리브레이션 데이터셋
    tokenizer=tokenizer,
    group_size=128,              # 양자화 그룹 크기
    desc_act=True,               # Activation order 사용
)

# 3. 양자화 수행 (시간이 걸림)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=gptq_config,
    device_map="auto"
)

# 4. 양자화된 모델 저장
model.save_pretrained("./qwen2.5-1.5b-gptq-4bit")
tokenizer.save_pretrained("./qwen2.5-1.5b-gptq-4bit")
'''
print(gptq_code)

print("\n💡 GPTQ 양자화는 1.5B 모델 기준 약 10~30분 소요됩니다.")
print("💡 실습에서는 이미 양자화된 커뮤니티 모델을 활용하는 것을 권장합니다.")

In [ ]:
# 📦 방법 2: 이미 양자화된 GPTQ 모델 로딩
print("📦 사전 양자화된 GPTQ 모델 로딩 방법")
print("=" * 60)

gptq_load_code = '''
from transformers import AutoModelForCausalLM, AutoTokenizer

# HuggingFace에서 GPTQ 양자화된 모델 검색:
# https://huggingface.co/models?search=qwen2.5+gptq

# 예시: 커뮤니티에서 양자화된 모델 로딩
model_id = "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4"  # 예시 경로

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto"
)

# 추론
messages = [{"role": "user", "content": "안녕하세요"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
'''
print(gptq_load_code)

print("💡 GPTQ 모델은 로딩이 빠르고, 추론 시 최적화 커널(ExLlama)을 자동으로 사용합니다.")
print("💡 HuggingFace에서 'GPTQ' 태그로 양자화된 모델을 검색할 수 있습니다.")

---

## 5️⃣ AWQ 양자화 개요

**AWQ (Activation-Aware Weight Quantization)**는 활성화(activation) 분포를 고려하여
중요한 가중치를 보존하는 양자화 기법입니다.

### AWQ 핵심 아이디어
- 🔹 모든 가중치가 동일하게 중요하지 않음
- 🔹 **활성화 크기가 큰 채널**의 가중치가 더 중요
- 🔹 중요한 가중치는 **스케일링 팩터**를 적용하여 보존
- 🔹 하드웨어 효율적인 양자화

### AWQ vs GPTQ

| 항목 | GPTQ | AWQ |
|------|------|-----|
| 접근법 | 레이어별 Hessian 최적화 | 활성화 기반 가중치 보존 |
| 양자화 속도 | 느림 | 빠름 |
| 추론 속도 | 빠름 | 매우 빠름 |
| 품질 | 좋음 | 좋음 (특히 4bit) |
| 하드웨어 지원 | 넓음 | GPU 최적화 |

In [ ]:
# 🔧 AWQ 양자화 코드 예시
print("🔧 AWQ 양자화 코드 예시")
print("=" * 60)

awq_code = '''
# AWQ 양자화 (autoawq 패키지 필요)
# pip install autoawq

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_path = "Qwen/Qwen2.5-1.5B-Instruct"
quant_path = "./qwen2.5-1.5b-awq-4bit"

# 1. 모델 로딩
model = AutoAWQForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 2. AWQ 양자화 설정
quant_config = {
    "zero_point": True,     # Zero-point 양자화
    "q_group_size": 128,    # 양자화 그룹 크기
    "w_bit": 4,             # 4bit 양자화
    "version": "GEMM"       # GEMM 커널 사용
}

# 3. 양자화 수행
model.quantize(tokenizer, quant_config=quant_config)

# 4. 저장
model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)
print(f"AWQ 양자화 모델 저장 완료: {quant_path}")

# 5. 양자화된 모델 로딩 (Transformers 통합)
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    quant_path,
    device_map="auto"
)
'''
print(awq_code)

print("\n💡 AWQ는 양자화 속도가 GPTQ보다 빠르고, 추론 성능도 우수합니다.")
print("💡 vLLM과의 호환성이 좋아 프로덕션 환경에서 많이 사용됩니다.")
print("💡 HuggingFace에서 'AWQ' 태그로 양자화된 모델을 검색할 수 있습니다.")

---

## 6️⃣ GGUF 포맷과 llama.cpp

**GGUF (GPT-Generated Unified Format)**는 llama.cpp에서 사용하는 모델 포맷으로,
**CPU 추론**에 최적화되어 있습니다.

### GGUF 특징
- 🔹 **CPU 추론 가능**: GPU 없이도 LLM 실행
- 🔹 **다양한 양자화 레벨**: Q2_K, Q3_K, Q4_K, Q5_K, Q6_K, Q8_0 등
- 🔹 **Ollama 호환**: Ollama에서 바로 사용 가능
- 🔹 **메모리 매핑(mmap)**: 디스크에서 직접 로딩 (빠른 시작)
- 🔹 **GPU 오프로딩**: 일부 레이어만 GPU에서 실행 가능

### GGUF 양자화 레벨

| 레벨 | 비트 | 크기(7B 기준) | 품질 |
|------|------|-------------|------|
| Q2_K | 2~3bit | ~2.8GB | 낮음 |
| Q3_K_M | 3~4bit | ~3.3GB | 보통 |
| **Q4_K_M** | **4~5bit** | **~4.1GB** | **권장** |
| Q5_K_M | 5~6bit | ~4.8GB | 좋음 |
| Q6_K | 6bit | ~5.5GB | 매우 좋음 |
| Q8_0 | 8bit | ~7.2GB | 최고 |

In [ ]:
# 🔧 GGUF 변환 및 사용 방법
print("🔧 GGUF 변환 및 사용 방법")
print("=" * 60)

gguf_guide = '''
# ============================================
# 방법 1: llama.cpp로 직접 변환
# ============================================

# 1. llama.cpp 설치
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
make

# 2. HuggingFace 모델 → GGUF 변환
python convert_hf_to_gguf.py \\
    --model Qwen/Qwen2.5-1.5B-Instruct \\
    --outfile qwen2.5-1.5b.gguf \\
    --outtype f16

# 3. 양자화
./llama-quantize qwen2.5-1.5b.gguf qwen2.5-1.5b-Q4_K_M.gguf Q4_K_M

# 4. 실행
./llama-cli -m qwen2.5-1.5b-Q4_K_M.gguf -p "인공지능이란" -n 100

# ============================================
# 방법 2: Ollama에서 사용
# ============================================

# Ollama는 GGUF 모델을 직접 로딩 가능
# Modelfile 작성:
# FROM ./qwen2.5-1.5b-Q4_K_M.gguf
# PARAMETER temperature 0.7
# SYSTEM "당신은 유용한 AI 어시스턴트입니다."

ollama create my-model -f Modelfile
ollama run my-model

# ============================================
# 방법 3: Python에서 llama-cpp-python 사용
# ============================================
# pip install llama-cpp-python
'''
print(gguf_guide)

In [ ]:
# 🐍 Python에서 GGUF 모델 사용 예시 (llama-cpp-python)
print("🐍 Python에서 GGUF 모델 사용 예시")
print("=" * 60)

gguf_python_code = '''
# pip install llama-cpp-python
from llama_cpp import Llama

# GGUF 모델 로딩
llm = Llama(
    model_path="./qwen2.5-1.5b-Q4_K_M.gguf",
    n_ctx=2048,         # 컨텍스트 길이
    n_gpu_layers=-1,    # 모든 레이어를 GPU에 올림 (-1)
    verbose=False
)

# 추론
output = llm(
    "인공지능의 미래에 대해 설명해주세요.",
    max_tokens=200,
    temperature=0.7,
    echo=False
)

print(output["choices"][0]["text"])

# Chat 형식
output = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
        {"role": "user", "content": "인공지능의 미래에 대해 설명해주세요."}
    ],
    max_tokens=200
)
print(output["choices"][0]["message"]["content"])
'''
print(gguf_python_code)

print("\n💡 GGUF는 Ollama의 기본 포맷이므로, Ollama와 함께 사용하는 것이 가장 편리합니다.")
print("💡 n_gpu_layers로 GPU 오프로딩 레이어 수를 조절할 수 있습니다.")

---

## 7️⃣ 양자화 방법별 성능 비교 (메모리, 속도, 품질)

### 비교 대상
- **BitsAndBytes**: 동적 양자화 (4bit NF4, 8bit)
- **GPTQ**: 사후 훈련 양자화 (Hessian 기반)
- **AWQ**: 활성화 인식 양자화
- **GGUF**: llama.cpp 양자화 (CPU/GPU 혼합)

### 종합 비교표

| 항목 | BnB 8bit | BnB 4bit | GPTQ 4bit | AWQ 4bit | GGUF Q4_K_M |
|------|----------|----------|-----------|----------|-------------|
| 메모리 | ~50% 절감 | ~75% 절감 | ~75% 절감 | ~75% 절감 | ~75% 절감 |
| 추론 속도 | 보통 | 보통 | 빠름 | 매우 빠름 | CPU 가능 |
| 품질 | 좋음 | 좋음 | 매우 좋음 | 매우 좋음 | 좋음 |
| 학습 지원 | QLoRA 가능 | QLoRA 가능 | 제한적 | X | X |
| 사용 편의성 | 매우 쉬움 | 매우 쉬움 | 중간 | 중간 | Ollama 통합 |
| vLLM 호환 | X | X | O | O | X |

In [ ]:
# 📊 양자화 방법별 성능 비교 시각화
import matplotlib.pyplot as plt
import numpy as np

# 비교 데이터 (대표적인 벤치마크 기준 - 상대값)
methods = ['FP16\n(baseline)', 'BnB\n8bit', 'BnB\n4bit', 'GPTQ\n4bit', 'AWQ\n4bit', 'GGUF\nQ4_K_M']

# 상대적 수치 (FP16 = 100 기준)
memory_usage = [100, 55, 30, 28, 28, 30]  # 메모리 사용량 (낮을수록 좋음)
inference_speed = [100, 85, 90, 120, 130, 70]  # 추론 속도 (높을수록 좋음)
quality = [100, 99, 97, 98, 98, 96]  # 품질 (높을수록 좋음)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']

# 메모리 사용량
bars1 = axes[0].bar(methods, memory_usage, color=colors, alpha=0.8)
axes[0].set_title('Memory Usage (Lower = Better)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Relative Memory (%)')
axes[0].axhline(y=100, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars1, memory_usage):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val}%', ha='center', va='bottom', fontsize=10)

# 추론 속도
bars2 = axes[1].bar(methods, inference_speed, color=colors, alpha=0.8)
axes[1].set_title('Inference Speed (Higher = Better)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Relative Speed (%)')
axes[1].axhline(y=100, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars2, inference_speed):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val}%', ha='center', va='bottom', fontsize=10)

# 품질
bars3 = axes[2].bar(methods, quality, color=colors, alpha=0.8)
axes[2].set_title('Quality (Higher = Better)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Relative Quality (%)')
axes[2].set_ylim(90, 102)
axes[2].axhline(y=100, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars3, quality):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                f'{val}%', ha='center', va='bottom', fontsize=10)

plt.suptitle('Quantization Methods Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('quantization_comparison.png', bbox_inches='tight', dpi=100)
plt.show()

print("✅ 양자화 방법별 비교 시각화 완료!")

In [ ]:
# 🔬 3B 모델 vs 1.5B 모델 양자화 메모리 비교 실습
print("🔬 모델 크기별 양자화 효과 비교")
print("=" * 60)

print("\n📌 1.5B 모델 (Qwen2.5-1.5B-Instruct) 4bit 로딩")
print_gpu_memory("로딩 전")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model_1_5b = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_1_5B,
    quantization_config=quantization_config,
    device_map="auto"
)
mem_1_5b_4bit = torch.cuda.memory_allocated() / 1024**3
print_gpu_memory("1.5B 4bit 로딩 후")

del model_1_5b
cleanup()

print("\n📌 3B 모델 (Qwen2.5-3B-Instruct) 4bit 로딩")
print_gpu_memory("로딩 전")

model_3b = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_3B,
    quantization_config=quantization_config,
    device_map="auto"
)
mem_3b_4bit = torch.cuda.memory_allocated() / 1024**3
print_gpu_memory("3B 4bit 로딩 후")

del model_3b
cleanup()

print("\n📊 4bit 양자화 메모리 비교:")
print(f"  - Qwen2.5-1.5B 4bit: {mem_1_5b_4bit:.2f}GB")
print(f"  - Qwen2.5-3B 4bit:   {mem_3b_4bit:.2f}GB")
print(f"  - RTX 4060 (8GB)에서 두 모델 모두 충분히 실행 가능!")

---

## 8️⃣ RTX 4060에서의 최적 양자화 전략

RTX 4060 (8GB VRAM) 환경에서의 실용적인 양자화 전략을 정리합니다.

### 용도별 권장 양자화 방법

| 용도 | 모델 크기 | 양자화 방법 | 메모리 사용 |
|------|----------|-----------|----------|
| 🔬 **QLoRA 학습** | 1.5B~3B | BnB 4bit (NF4) | ~2~4GB |
| 🔬 **QLoRA 학습** | 7B | BnB 4bit (NF4) | ~6~7GB |
| ⚡ **빠른 추론** | 1.5B~3B | GPTQ/AWQ 4bit | ~1~2GB |
| ⚡ **빠른 추론** | 7B | GPTQ/AWQ 4bit | ~4~5GB |
| 🖥️ **Ollama 서빙** | 3B~7B | GGUF Q4_K_M | ~2~5GB |
| 📊 **품질 우선** | 1.5B~3B | BnB 8bit | ~2~4GB |

### RTX 4060 실전 팁
- ✅ 학습: BitsAndBytes 4bit (QLoRA) 사용
- ✅ 추론: GPTQ 또는 AWQ 4bit 사용
- ✅ 로컬 서빙: Ollama + GGUF Q4_K_M
- ⚠️ FP16은 3B까지만 가능
- ⚠️ 7B FP16은 RTX 4060에서 불가능

In [ ]:
# 🎯 RTX 4060 최적 설정 가이드 코드
print("🎯 RTX 4060 환경별 최적 모델 로딩 코드")
print("=" * 60)

optimal_configs = {
    "QLoRA 학습용 (1.5B~7B)": '''
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True  # 메모리 추가 절약
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",  # 또는 7B
    quantization_config=bnb_config,
    device_map="auto"
)
''',
    "빠른 추론용 (GPTQ)": '''
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",  # 사전 양자화 모델
    device_map="auto"
)
''',
    "Ollama 서빙용": '''
# 터미널에서:
ollama pull qwen2.5:3b    # 3B Q4 모델 (~2GB)
ollama pull qwen2.5:7b    # 7B Q4 모델 (~4.7GB)
ollama run qwen2.5:3b     # 실행
'''
}

for title, code in optimal_configs.items():
    print(f"\n📌 {title}")
    print("-" * 60)
    print(code)

print("\n💡 핵심: RTX 4060에서는 4bit 양자화가 필수입니다!")
print("💡 학습에는 BnB, 추론에는 GPTQ/AWQ, 서빙에는 GGUF를 사용하세요.")

In [ ]:
# 📊 양자화 의사결정 플로우차트
print("📊 양자화 방법 선택 가이드")
print("=" * 60)

decision_tree = """
    [목적이 무엇인가요?]
         |
    ┌────┼────────────┐
    │    │            │
  학습  추론        서빙
    │    │            │
    │    │            │
  QLoRA │         Ollama?
  사용  │          ┌──┴──┐
    │    │         예    아니오
    │    │         │      │
   BnB   │       GGUF   vLLM?
   4bit  │      Q4_K_M  ┌──┴──┐
    │    │              예    아니오
    │    │              │      │
    │  GPU만?        AWQ    GPTQ
    │  ┌──┴──┐       4bit   4bit
    │ 예    아니오
    │  │      │
    │ GPTQ  GGUF
    │ AWQ   (CPU)
    │
   ✅ BitsAndBytes 4bit NF4
"""
print(decision_tree)

print("💡 대부분의 경우 BitsAndBytes 4bit (NF4)가 가장 범용적입니다.")
print("💡 프로덕션 서빙에는 AWQ + vLLM 조합이 최적입니다.")

---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **양자화의 필요성**: 메모리 절감, 속도 향상, 접근성 확대

2️⃣ **정밀도별 비교**: FP32 → FP16 → INT8 → INT4로 갈수록 메모리 절감

3️⃣ **BitsAndBytes**: 가장 간편한 동적 양자화, QLoRA 학습에 필수

4️⃣ **GPTQ**: 캘리브레이션 기반 고품질 양자화, 빠른 추론

5️⃣ **AWQ**: 활성화 인식 양자화, vLLM과 호환성 우수

6️⃣ **GGUF**: llama.cpp/Ollama 포맷, CPU 추론 가능

7️⃣ **성능 비교**: 4bit 양자화로 메모리 75% 절감, 품질 저하 미미

8️⃣ **RTX 4060 전략**: 학습은 BnB, 추론은 GPTQ/AWQ, 서빙은 GGUF

### 핵심 키워드
- `BitsAndBytesConfig(load_in_4bit=True)` - 가장 많이 사용하는 양자화 설정
- `bnb_4bit_quant_type="nf4"` - QLoRA 최적 양자화 타입
- `bnb_4bit_use_double_quant=True` - 추가 메모리 절약

### 다음 세션 예고
- 🔜 **Session 23**: GPTQ, AWQ, QLoRA 비교 실습

In [ ]:
# 🧹 최종 메모리 정리
cleanup()

print("\n" + "=" * 60)
print("✅ Session 22: 양자화 (Quantization) 개념 완료!")
print("=" * 60)
print("\n📌 다음 세션(Session 23): GPTQ, AWQ, QLoRA 비교 실습")
print("📌 준비물: auto-gptq, autoawq 패키지 설치")
print("   pip install auto-gptq autoawq")